In [1]:
# Retry simulation with bug fix for transient RFI addition
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta


In [2]:

# ----------------------- Parameters -----------------------
f0 = 1.75e9      # start freq (Hz)
f1 = 3.5e9       # end freq (Hz)
Nfreq = 2048     # number of frequency channels
Ntime = 1024     # number of time samples
t_int = 1.0      # integration time per sample (s) - arbitrary for simulation
srctime_amp = 5.0   # amplitude of continuum source (relative units)
thermal_noise_std = 1.0  # thermal noise standard deviation
spectral_index = -0.7    # continuum spectral index (S ~ nu^alpha)
line_center = 2.2e9      # spectral line center (Hz)
line_width_hz = 1e6      # line width (Hz)
line_amp = 20.0          # amplitude of spectral line above continuum (in same units)

# RFI definitions
rfi_narrow_freqs = [2.05e9, 2.45e9, 3.0e9]   # Hz - persistent narrowband RFI
rfi_narrow_amp = 100.0
rfi_burst_times = [(200, 240), (700, 710)]   # time index ranges for broadband bursts
rfi_burst_amp = 50.0
rfi_transient_time = 500                     # single-time transient burst (time index)
rfi_transient_amp = 200.0
rfi_narrow_width_hz = 100e3  # narrow RFI width (Hz)

# Bandpass ripple + gain drift
ripple_amp = 0.05     # fractional ripple amplitude
ripple_period_hz = 5e7  # ripple period in Hz
gain_drift_amp = 0.1  # fractional gain drift over time (slow)

# ----------------------- Build axes -----------------------
freqs = np.linspace(f0, f1, Nfreq)  # in Hz
times = np.arange(Ntime) * t_int    # in seconds
freqs_ghz = freqs / 1e9

# ----------------------- Create clean sky model -----------------------
continuum_spec = (freqs / 1e9) ** spectral_index   # normalized shape (arbitrary normalization)
continuum_spec *= srctime_amp / np.mean(continuum_spec)  # set average continuum amplitude ~ srctime_amp

line_profile = line_amp * np.exp(-0.5 * ((freqs - line_center) / line_width_hz) ** 2)

sky_spec = continuum_spec + line_profile

dynspec = np.tile(sky_spec[np.newaxis, :], (Ntime, 1)).astype(np.float32)

# ----------------------- Apply bandpass ripple (frequency-dependent multiplicative) -----------------------
bandpass_ripple = 1.0 + ripple_amp * np.sin(2 * np.pi * freqs / ripple_period_hz)
dynspec *= bandpass_ripple[np.newaxis, :]

# ----------------------- Apply slow gain drift (time-dependent multiplicative) -----------------------
gain_drift = 1.0 + gain_drift_amp * 0.5 * (1 + np.sin(2 * np.pi * times / (Ntime * 4)))  # very slow sinusoidal drift
dynspec *= gain_drift[:, np.newaxis]

# ----------------------- Add thermal noise -----------------------
noise = np.random.normal(loc=0.0, scale=thermal_noise_std, size=(Ntime, Nfreq))
dynspec += noise

# ----------------------- Add narrowband persistent RFI -----------------------
for rf in rfi_narrow_freqs:
    center_idx = np.argmin(np.abs(freqs - rf))
    width_pix = int(np.ceil(rfi_narrow_width_hz / (freqs[1] - freqs[0])))  # convert width Hz -> pixels
    tb = rfi_narrow_amp * (1.0 + 0.5 * np.sin(2 * np.pi * times / 50.0))[:, np.newaxis]
    gaussian_freq = np.exp(-0.5 * ((np.arange(Nfreq) - center_idx) / max(1, width_pix))**2)
    dynspec += tb * gaussian_freq[np.newaxis, :]

# ----------------------- Add broadband bursts (in time ranges) -----------------------
for t0, t1 in rfi_burst_times:
    t0 = max(0, t0); t1 = min(Ntime, t1)
    dynspec[t0:t1, :] += rfi_burst_amp * (1.0 + 0.2 * np.random.randn(t1-t0, Nfreq))

# ----------------------- Add single-time transient RFI (wideband) -----------------------
transient_idx = int(np.clip(rfi_transient_time, 0, Ntime-1))
# create a wideband transient shape across frequency (e.g., gaussian across freq or flat); we use a flat strong spike with small freq structure
dynspec[transient_idx, :] += rfi_transient_amp * (1.0 + 0.1 * np.random.randn(Nfreq))

# # ----------------------- Save simulation -----------------------
# out_dir = "/mnt/data"
# os.makedirs(out_dir, exist_ok=True)
# out_path = os.path.join(out_dir, "sim_radiotelescope.npz")
# np.savez_compressed(out_path, dynspec=dynspec, freqs=freqs, times=times, freqs_ghz=freqs_ghz)
# print("Saved simulated dynamic spectrum to:", out_path)


In [ ]:

# ----------------------- Plot: Waterfall -----------------------
fig1 = plt.figure(figsize=(10, 6))
plt.imshow(dynspec, aspect='auto', origin='lower', extent=[freqs_ghz[0], freqs_ghz[-1], times[0], times[-1]])
plt.xlabel("Frequency (GHz)")
plt.ylabel("Time (s)")
plt.title("Simulated dynamic spectrum (waterfall)")
plt.colorbar(label="Amplitude (arb. units)")
plt.tight_layout()
p1 = os.path.join(out_dir, "waterfall.png")
plt.savefig(p1)
plt.show()

# ----------------------- Plot: Mean Spectrum -----------------------
mean_spec = dynspec.mean(axis=0)
fig2 = plt.figure(figsize=(10, 4))
plt.plot(freqs_ghz, mean_spec)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Mean amplitude (arb. units)")
plt.title("Mean spectrum (time-averaged)")
plt.grid(True)
plt.tight_layout()
p2 = os.path.join(out_dir, "mean_spectrum.png")
plt.savefig(p2)
plt.show()

# ----------------------- Plot: Time series at a selected frequency (near spectral line) -----------------------
idx_line = np.argmin(np.abs(freqs - line_center))
fig3 = plt.figure(figsize=(10, 3))
plt.plot(times, dynspec[:, idx_line])
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (arb. units)")
plt.title(f"Time series at {freqs_ghz[idx_line]:.4f} GHz (near simulated spectral line)")
plt.tight_layout()
p3 = os.path.join(out_dir, "time_series_linechan.png")
plt.savefig(p3)
plt.show()

# ----------------------- Done -----------------------
print("Plots saved to:", p1, p2, p3)
print("You can download the npz file and the PNGs from /mnt/data in this environment.")
